# 🎨 Cartoonify — Satirical Image Transformer

Transforms any photo into a cartoon/satirical illustration using:
- **FLUX.1-dev** — base diffusion model
- **ControlNet Depth** — preserves scene composition from the source
- **LoRA fine-tune** — applies the cartoon/satirical style

**Requires:** Google Colab Pro → Runtime → Change runtime type → **A100 GPU**

---

## 1 — Confirm A100 GPU

You need an A100 (40 GB VRAM) to run FLUX.1-dev. Confirm below before proceeding.

In [1]:
!nvidia-smi

Sat Jun 13 00:16:57 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P0             44W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 2 — Install Dependencies

In [2]:
!pip install --quiet diffusers transformers accelerate sentencepiece
!pip install --quiet gradio
!pip install --quiet huggingface_hub

In [ ]:
# Restart runtime so installed packages are picked up cleanly
import os
os.kill(os.getpid(), 9)
print("IGNORE: session crashed. This is intentional — continue to the next cell.")

## 3 — Imports

In [1]:
import gc
import datetime
import os
import numpy as np
import torch
import gradio as gr
from PIL import Image
from diffusers import FluxControlNetPipeline, FluxControlNetModel
from transformers import pipeline as hf_pipeline

print(f'torch  : {torch.__version__}')
print(f'CUDA   : {torch.cuda.is_available()} — {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')
print(f'gradio : {gr.__version__}')

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


torch  : 2.11.0+cu128
CUDA   : True — NVIDIA A100-SXM4-40GB
gradio : 5.50.0


## 4 — Mount Google Drive

Used to persist LoRA weights and auto-save generated images.

In [2]:
import os, shutil
from google.colab import drive

# Clear stale mountpoint left over from previous session restart
if os.path.isdir('/content/drive') and os.listdir('/content/drive'):
    os.system('fusermount -u /content/drive 2>/dev/null || true')
    shutil.rmtree('/content/drive', ignore_errors=True)

drive.mount('/content/drive')

Mounted at /content/drive


## 5 — Hugging Face Token

FLUX.1-dev is a gated model — you must accept the licence on HuggingFace before it can be downloaded.

1. Sign up at [huggingface.co](https://huggingface.co) and accept the [FLUX.1-dev licence](https://huggingface.co/black-forest-labs/FLUX.1-dev)
2. Create a **Read** token at `huggingface.co/settings/tokens`
3. In Colab: click 🔑 **Secrets** (left sidebar) → Name: `HF_TOKEN` → paste token → enable Notebook access

In [3]:
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get('HF_TOKEN')
login(token=HF_TOKEN, add_to_git_credential=False)
print('✓ Logged in to Hugging Face')

✓ Logged in to Hugging Face


## 6 — Configuration

**This is the only cell you need to edit** when swapping the LoRA or changing defaults.

| Variable | What it does |
|---|---|
| `LORA_SOURCE` | `"huggingface"` = download from HF · `"drive"` = load local file |
| `LORA_HF_REPO` | HuggingFace repo ID for the LoRA |
| `LORA_HF_WEIGHT` | Filename of the `.safetensors` weight inside the repo |
| `LORA_DRIVE_PATH` | Full path to your `.safetensors` on Drive (used when `LORA_SOURCE = "drive"`) |
| `DEFAULT_TRIGGER` | The keyword baked into training captions — also editable live in the UI |

In [4]:
# ── LoRA source ────────────────────────────────────────────────────────────────
LORA_SOURCE      = 'drive'   # 'huggingface' | 'drive'

# Placeholder LoRA (Ghibli cartoon style) — replace with your trained model
LORA_HF_REPO     = 'strangerzonehf/Ghibli-Flux-Cartoon-LoRA'
LORA_HF_WEIGHT   = 'Ghibili-Cartoon-Art.safetensors'

# Your trained LoRA on Drive (set LORA_SOURCE = 'drive' to use this)
LORA_DRIVE_PATH  = '/content/drive/MyDrive/cartoonify/02_FLUX.1/weights_FLUX.1/gdo_cartoon/gdo_cartoon.safetensors'

# ── Trigger word ───────────────────────────────────────────────────────────────
# The keyword used in training captions. Also editable live in the UI.
DEFAULT_TRIGGER  = 'gdo_cartoon'

# ── Prompt default ─────────────────────────────────────────────────────────────
DEFAULT_PROMPT   = 'satirical cartoon illustration, bold outlines, vivid flat colours, expressive exaggerated features'

# ── Generation defaults (all overridable from UI sliders) ──────────────────────
DEFAULT_GUIDANCE = 3.5
DEFAULT_STEPS    = 28
DEFAULT_CN_SCALE = 0.8   # Shakker-Labs recommends 0.8 for Union Pro 2.0
DEFAULT_SEED     = 42

# ── Base models ────────────────────────────────────────────────────────────────
BASE_MODEL       = 'black-forest-labs/FLUX.1-dev'
CONTROLNET_MODEL = 'Shakker-Labs/FLUX.1-dev-ControlNet-Union-Pro-2.0'   # fixed: dropped wrong -fp16 suffix
DEPTH_MODEL      = 'depth-anything/Depth-Anything-V2-Small-hf'

# ── Output directory ───────────────────────────────────────────────────────────
OUTPUT_DIR = '/content/drive/MyDrive/cartoonify/outputs'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f'LoRA source  : {LORA_SOURCE}')
print(f'LoRA         : {LORA_HF_REPO if LORA_SOURCE == "huggingface" else LORA_DRIVE_PATH}')
print(f'Trigger word : "{DEFAULT_TRIGGER}"')
print(f'Output dir   : {OUTPUT_DIR}')

LoRA source  : drive
LoRA         : /content/drive/MyDrive/cartoonify/02_FLUX.1/weights_FLUX.1/gdo_cartoon/gdo_cartoon.safetensors
Trigger word : "gdo_cartoon"
Output dir   : /content/drive/MyDrive/cartoonify/outputs


## 7 — Load Models

First run downloads ~24 GB (FLUX.1-dev + ControlNet). Subsequent runs use the Colab cache — much faster.

⏱️ Expected time: **3–6 minutes** on first run, **~1 minute** on warm cache.

In [5]:
import gc

for _name in ['pipe', 'controlnet', 'depth_estimator']:
    if _name in globals():
        del globals()[_name]
gc.collect()
torch.cuda.empty_cache()
torch.cuda.synchronize()
_free, _total = torch.cuda.mem_get_info()
print(f'GPU before loading: {_free/1e9:.1f} GB free / {_total/1e9:.1f} GB total\n')

print('Loading depth estimator (CPU)...')
depth_estimator = hf_pipeline('depth-estimation', model=DEPTH_MODEL, device='cpu')
print('✓ Depth estimator ready')

print('\nLoading ControlNet...')
controlnet = FluxControlNetModel.from_pretrained(CONTROLNET_MODEL, torch_dtype=torch.bfloat16)
print('✓ ControlNet ready')

print('\nLoading FLUX.1-dev pipeline...')
pipe = FluxControlNetPipeline.from_pretrained(BASE_MODEL, controlnet=controlnet, torch_dtype=torch.bfloat16).to('cuda')
print('✓ FLUX pipeline ready')

import peft.import_utils as _peft_utils
import peft.tuners.lora.torchao as _peft_torchao
_orig = _peft_utils.is_torchao_available
def _safe():
    try: return _orig()
    except ImportError: return False
_peft_utils.is_torchao_available = _safe
_peft_torchao.is_torchao_available = _safe
print('✓ PEFT torchao patched')

print('\nLoading LoRA weights...')
if LORA_SOURCE == 'huggingface':
    pipe.load_lora_weights(LORA_HF_REPO, weight_name=LORA_HF_WEIGHT)
else:
    pipe.load_lora_weights(LORA_DRIVE_PATH)
print('✅ All models ready.')


GPU before loading: 42.0 GB free / 42.4 GB total

Loading depth estimator (CPU)...


config.json:   0%|          | 0.00/950 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/99.2M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/287 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/219 [00:00<?, ?it/s]

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

✓ FLUX pipeline ready
✓ PEFT torchao patched

Loading LoRA weights...


No LoRA keys associated to CLIPTextModel found with the prefix='text_encoder'. This is safe to ignore if LoRA state dict didn't originally have any CLIPTextModel related params. You can also try specifying `prefix=None` to resolve the warning. Otherwise, open an issue if you think it's unexpected: https://github.com/huggingface/diffusers/issues/new


✅ All models ready.


## 8 — Launch Interface

Run this cell to start the Gradio app. A **public URL** will appear below — open it in any browser.

The interface stays live as long as this Colab session is running. Re-run this cell to restart the UI without reloading models.

In [6]:
# ── Inference function ─────────────────────────────────────────────────────────

def cartoonify(
    image,
    prompt,
    trigger_word,
    guidance_scale,
    num_steps,
    cn_scale,
    seed,
    progress=gr.Progress(track_tqdm=True),
):
    if image is None:
        raise gr.Error('Please upload an image first.')

    # Build full prompt: prepend trigger word if provided
    trigger = trigger_word.strip()
    style   = prompt.strip()
    full_prompt = f'{trigger} {style}'.strip() if trigger else style

    # Resize source to fixed 1024x1024
    src = Image.fromarray(image).convert('RGB')
    src = src.resize((1024, 1024), Image.LANCZOS)

    # Generate depth map (used as ControlNet input)
    progress(0.0, desc='Generating depth map...')
    depth_out  = depth_estimator(src)
    depth_arr  = np.array(depth_out['depth'])
    d_min, d_max = depth_arr.min(), depth_arr.max()
    if d_max > d_min:
        depth_norm = ((depth_arr - d_min) / (d_max - d_min) * 255).astype(np.uint8)
    else:
        depth_norm = np.zeros_like(depth_arr, dtype=np.uint8)
    depth_rgb = Image.fromarray(depth_norm).convert('RGB')

    # Run FLUX inference
    progress(0.15, desc='Running FLUX inference (~30 seconds)...')
    generator = torch.Generator(device='cuda').manual_seed(int(seed))

    with torch.inference_mode():
        result = pipe(
            prompt=full_prompt,
            control_image=depth_rgb,
            control_mode=2,
            width=1024,
            height=1024,
            num_inference_steps=int(num_steps),
            guidance_scale=float(guidance_scale),
            controlnet_conditioning_scale=float(cn_scale),
            generator=generator,
            joint_attention_kwargs={'scale': 1.0},
            max_sequence_length=512,
        ).images[0]

    # Auto-save to Drive
    ts = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    save_path = f'{OUTPUT_DIR}/{ts}_cartoonify.png'
    result.save(save_path)
    print(f'Saved: {save_path}')

    gc.collect()
    torch.cuda.empty_cache()

    return depth_rgb, result

In [7]:
# ── CSS ────────────────────────────────────────────────────────────────────────

CSS = '''
/* ── Global ──────────────────────────────────────────────────────── */
.gradio-container {
    font-family: "Inter", -apple-system, BlinkMacSystemFont, sans-serif !important;
    background: #f0eff5 !important;
    max-width: 1280px !important;
    margin: 0 auto !important;
    padding: 1.5rem !important;
}

/* ── App header ───────────────────────────────────────────────────── */
#app-header {
    background: linear-gradient(135deg, #0f0c29 0%, #302b63 55%, #24243e 100%);
    border-radius: 18px;
    padding: 2.5rem 2rem 2.2rem;
    text-align: center;
    margin-bottom: 1.5rem;
    box-shadow: 0 8px 30px rgba(48, 43, 99, 0.35);
}
.app-title {
    font-size: 3rem;
    font-weight: 900;
    letter-spacing: -2px;
    color: #ffffff;
    margin: 0 0 0.5rem 0;
    line-height: 1;
    display: block;
}
.app-subtitle {
    font-size: 1rem;
    color: #c4b5fd;
    margin: 0;
    font-weight: 400;
    line-height: 1.6;
    display: block;
}
.app-badge {
    display: inline-block;
    margin-top: 1rem;
    background: rgba(255,255,255,0.12);
    border: 1px solid rgba(255,255,255,0.2);
    border-radius: 999px;
    padding: 0.25rem 0.85rem;
    font-size: 0.75rem;
    color: #e9d5ff;
    letter-spacing: 0.05em;
    font-weight: 600;
    text-transform: uppercase;
}

/* ── Cards ────────────────────────────────────────────────────────── */
.input-card, .output-card {
    background: #ffffff;
    border-radius: 16px;
    padding: 1.5rem 1.5rem 1.75rem;
    box-shadow: 0 1px 4px rgba(0,0,0,0.07);
    border: 1px solid #e5e7eb;
}

/* ── Section labels ───────────────────────────────────────────────── */
.section-label {
    display: block;
    font-size: 0.68rem;
    font-weight: 800;
    text-transform: uppercase;
    letter-spacing: 0.12em;
    color: #7c3aed;
    margin-bottom: 0.6rem;
    margin-top: 1.1rem;
}
.section-label:first-child { margin-top: 0; }

/* ── Upload zone ──────────────────────────────────────────────────── */
.upload-zone {
    border: 2.5px dashed #7c3aed !important;
    border-radius: 14px !important;
    background: #faf5ff !important;
    transition: border-color 0.2s, background 0.2s !important;
}
.upload-zone:hover {
    border-color: #5b21b6 !important;
    background: #f5edff !important;
}

/* ── Generate button ──────────────────────────────────────────────── */
#generate-btn {
    background: linear-gradient(135deg, #7c3aed 0%, #5b21b6 100%) !important;
    color: white !important;
    border: none !important;
    border-radius: 12px !important;
    font-size: 1.05rem !important;
    font-weight: 700 !important;
    height: 54px !important;
    width: 100% !important;
    letter-spacing: 0.02em !important;
    cursor: pointer !important;
    box-shadow: 0 4px 18px rgba(124, 58, 237, 0.38) !important;
    transition: opacity 0.2s, transform 0.15s !important;
    margin-top: 1rem !important;
}
#generate-btn:hover {
    opacity: 0.90 !important;
    transform: translateY(-1px) !important;
}
#generate-btn:active { transform: translateY(0) !important; }
#generate-btn:disabled { opacity: 0.5 !important; cursor: wait !important; }

/* ── Output image ─────────────────────────────────────────────────── */
.output-image {
    border-radius: 12px !important;
    overflow: hidden !important;
    background: #f9fafb !important;
}

/* ── Tip box ──────────────────────────────────────────────────────── */
.tip-box {
    background: #f5f3ff;
    border: 1px solid #ddd6fe;
    border-radius: 10px;
    padding: 0.75rem 1rem;
    font-size: 0.82rem;
    color: #5b21b6;
    line-height: 1.5;
    margin-bottom: 0.75rem;
}

/* ── Hide Gradio footer ───────────────────────────────────────────── */
footer { display: none !important; }
'''


# ── Gradio UI ──────────────────────────────────────────────────────────────────

with gr.Blocks(css=CSS, theme=gr.themes.Soft(), title='Cartoonify') as demo:

    # ── Header ────────────────────────────────────────────────────────────────
    gr.HTML('''
        <div id="app-header">
            <span class="app-title">🎨 Cartoonify</span>
            <span class="app-subtitle">
                Drop a photo &mdash; get a satirical cartoon.<br>
                FLUX.1-dev &plus; ControlNet Depth &plus; LoRA
            </span>
            <span class="app-badge">Prototype &nbsp;&middot;&nbsp; Single-user</span>
        </div>
    ''')

    with gr.Row(equal_height=False, variant='panel'):

        # ── Left column — Inputs ───────────────────────────────────────────────
        with gr.Column(scale=5, min_width=360, elem_classes=['input-card']):

            gr.HTML('<span class="section-label">📤 Source Image</span>')
            img_input = gr.Image(
                label='Drag & drop a photo here, or click to browse',
                type='numpy',
                elem_classes=['upload-zone'],
                sources=['upload', 'clipboard'],
                height=250,
            )

            gr.HTML('<span class="section-label">✍️ Style Prompt</span>')
            prompt_input = gr.Textbox(
                label='Describe the cartoon style you want',
                placeholder='e.g. satirical cartoon, bold outlines, vivid flat colours, exaggerated expressions',
                value=DEFAULT_PROMPT,
                lines=3,
                max_lines=6,
            )

            gr.HTML('<span class="section-label">🔑 LoRA Trigger Word</span>')
            trigger_input = gr.Textbox(
                label='Trigger word (editable — no restart needed)',
                placeholder='e.g. Ghibli Art — leave blank if your LoRA has no trigger',
                value=DEFAULT_TRIGGER,
                lines=1,
                info='The keyword baked into your LoRA training captions. Update here when you swap models.',
            )

            gr.HTML('<span class="section-label">⚙️ Advanced Controls</span>')
            with gr.Accordion('Expand to adjust generation parameters', open=False):
                gr.HTML('<div class="tip-box">💡 Default values work well for most images. Try adjusting ControlNet Scale first if the composition feels off.</div>')
                guidance_slider = gr.Slider(
                    minimum=1.0, maximum=15.0, value=DEFAULT_GUIDANCE,
                    step=0.5, label='Guidance Scale',
                    info='How strongly the prompt steers the output. 3–5 is the FLUX sweet spot.',
                )
                steps_slider = gr.Slider(
                    minimum=10, maximum=50, value=DEFAULT_STEPS,
                    step=1, label='Inference Steps',
                    info='More steps = better quality, slower generation. 28 is a good balance.',
                )
                cn_slider = gr.Slider(
                    minimum=0.1, maximum=2.0, value=DEFAULT_CN_SCALE,
                    step=0.05, label='ControlNet Scale',
                    info='How strictly depth structure is enforced. Lower = more creative freedom.',
                )
                seed_input = gr.Number(
                    value=DEFAULT_SEED, label='Seed',
                    precision=0,
                    info='Same seed + same inputs = identical result. Change to explore variations.',
                )

            generate_btn = gr.Button(
                '🎨  Cartoonify  →',
                variant='primary',
                elem_id='generate-btn',
            )

        # ── Right column — Outputs ─────────────────────────────────────────────
        with gr.Column(scale=6, min_width=480, elem_classes=['output-card']):

            gr.HTML('<span class="section-label">🎭 Cartoonified Result</span>')
            result_output = gr.Image(
                label='Output  ·  Use the ↓ button to download',
                type='pil',
                elem_classes=['output-image'],
                show_download_button=True,
                height=480,
                interactive=False,
            )

            with gr.Accordion('🗺️ Depth Control Map  (technical preview)', open=False):
                gr.HTML('<div class="tip-box" style="margin-top:0.5rem;">This is the depth map used as a ControlNet conditioning signal — it encodes near/far distance in the scene, guiding composition without dictating pixel details.</div>')
                depth_output = gr.Image(
                    label='Depth map',
                    type='pil',
                    elem_classes=['output-image'],
                    interactive=False,
                    height=220,
                )

    # ── Button wiring ──────────────────────────────────────────────────────────
    generate_btn.click(
        fn=cartoonify,
        inputs=[
            img_input,
            prompt_input,
            trigger_input,
            guidance_slider,
            steps_slider,
            cn_slider,
            seed_input,
        ],
        outputs=[depth_output, result_output],
        api_name='cartoonify',
    )


demo.launch(share=True, show_error=True, quiet=False)

/tmp/ipykernel_834/2276726118.py:136: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, theme=gr.themes.Soft(), title='Cartoonify') as demo:
/tmp/ipykernel_834/2276726118.py:136: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CSS, theme=gr.themes.Soft(), title='Cartoonify') as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://00ef9124480d3b1758.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
